# Lekcja 17 - Tworzenie lokalnych agentów AI z Foundry Local i Qwen

W tym notatniku zbudujesz **lokalnego asystenta inżynierskiego**, który działa w całości na Twojej stacji roboczej. Ani razu nie jest używane przetwarzanie w chmurze. Asystent potrafi:

1. **Wywoływać narzędzia** za pomocą wywoływania funkcji Qwen przez Foundry Local.
2. **Wypisywać i czytać pliki** wewnątrz sandboxowego katalogu projektu.
3. **Analizować kod** za pomocą prostych metryk.
4. **Przeszukiwać dokumentację** przy użyciu lokalnego RAG (Chroma).
5. **Korzystać z lokalnego serwera MCP** (pomijane łagodnie, jeśli nie jest skonfigurowany).

Kod agenta wygląda prawie identycznie jak w lekcjach chmurowych – jedyna zmiana to zmiana punktu końcowego klienta z chmury na `localhost`.


## Konfiguracja

Przed uruchomieniem tego notatnika:

1. **Zainstaluj Microsoft Foundry Local** (zobacz [dokumentację](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) dla swojego systemu operacyjnego).
2. **Pobierz i uruchom model Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Zainstaluj poniższe pakiety Pythona.

Wszystko działa lokalnie. Maszyna z około 8 GB RAM to realistyczne minimum.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Połącz się z Foundry Local

`FoundryLocalManager` pobiera model, jeśli jest to konieczne, uruchamia lokalną usługę i udostępnia nam **punkt końcowy zgodny z OpenAI**. Następnie wskazujemy na niego standardowe SDK OpenAI. Klucz API jest lokalnym zastępczym — nie jest używane żadne poświadczenie chmurowe.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Narzędzia lokalne (piaskownica operacji na plikach)

Tworzymy na dysku mały przykładowy projekt, a następnie definiujemy narzędzia ograniczone do katalogu głównego tego projektu. Sprawdzenie piaskownicy ma znaczenie nawet lokalnie: narzędzie, które odczytuje dowolne ścieżki, działa z uprawnieniami użytkownika i może mieć dostęp do wszystkiego, co ty możesz.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Lokalny RAG z Chroma

Osadzamy niewielki zestaw fragmentów dokumentacji w lokalnej kolekcji Chroma. Chroma działa w procesie i przechowuje wektory na dysku — bez serwera, bez chmury. Narzędzie `search_docs` pobiera najbardziej istotne fragmenty dla zapytania.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Pętla wywoływania narzędzi

Teraz rejestrujemy narzędzia w modelu, korzystając ze schematu narzędzi OpenAI i uruchamiamy standardową pętlę wywoływania narzędzi — model prosi o narzędzie, wykonujemy je lokalnie, przekazujemy wynik z powrotem i powtarzamy, aż model wygeneruje ostateczną odpowiedź. Niezawodne wywoływanie funkcji w Qwen sprawia, że działa to na urządzeniu.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Lokalny MCP (opcjonalnie)

MCP jest transportem, a nie usługą w chmurze — serwer MCP może działać jako lokalny proces przez `stdio`. Poniższa komórka pokazuje, jak połączyć się z lokalnym serwerem MCP, jeśli masz tak skonfigurowany (na przykład serwer systemu plików). Pomija to w sposób łagodny, gdy `LOCAL_MCP_COMMAND` nie jest ustawiona, więc notatnik nadal działa od początku do końca bez niej.

Uwaga dotycząca bezpieczeństwa: lokalny serwer MCP działa z uprawnieniami twojego użytkownika. Ogranicz go do katalogu projektu i zweryfikuj jego wyniki przed podjęciem działania na ich podstawie.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Podsumowanie

Zbudowałeś asystenta inżynieryjnego, który działa w całości na Twoim komputerze:

- **Foundry Local** obsługiwał model **Qwen** za pomocą kompatybilnego z OpenAI endpointu — tak, aby kod agenta odpowiadał lekcjom z chmury.
- **Sandboxed tools** zapewniały agentowi dostęp do plików i analizę kodu bez wychodzenia z katalogu projektu.
- **Chroma** dostarczała **lokalny RAG** nad dokumentacją.
- **Local MCP** pokazywał, jak ponownie wykorzystać ekosystem MCP offline.

W żadnym momencie nie użyto chmurowego wnioskowania.

### Wyzwanie

Rozszerz lokalnego agenta, aby:

1. **Obsługiwał wiele serwerów MCP** — połącz serwer systemu plików i serwer git i pozwól agentowi wybierać między nimi.
2. **Używał lokalnej pamięci** — zachowuj krótką historię rozmowy na dysku, aby asystent pamiętał wcześniejsze interakcje przy ponownym uruchomieniu notatnika.
3. **Obsługiwał lokalną wieloagentową orkiestrację** — dodaj drugiego lokalnego agenta (np. recenzenta) i pozwól im współpracować przy zadaniu.

W następnej lekcji nauczysz się jak zabezpieczyć wdrożonych agentów AI.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
